# Loader SCGRec — reproducción (Tarea 0)

```
load_scgrec — Loader del dataset SCGRec (Yang et al. WWW'22) para reproducción
==============================================================================
Deja listas las INTERACCIONES (train/valid/test) y las CATEGORÍAS (género/dev/
publisher) en el formato del pipeline CPGRec+, para la Tarea 0 (reproducir →
extender). NO usa señal social (saltamos friends.txt / Groups.txt).

Formato confirmado: archivos **separados por COMAS, sin header**. En los splits,
cada línea = `user_id, game1, game2, ...` (user_id = steamid64); en *_time.txt
los tiempos van alineados y pueden ser `\\N` (faltante).

MEMORIA: el parser hace *streaming a parquet por lotes* (no acumula los ~95M en
RAM). Tipos compactos int64/int32/float32. Debería correr en Colab estándar.

Cómo correr en Colab:
  1) Ejecuta de arriba a abajo.
  2) Si igual te quedas corto, baja BATCH o usa MAX_TRAIN_LINES.
```

## Setup + descarga (si hace falta)

In [1]:
import os, ast, json
from itertools import zip_longest
from array import array
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

DATA_DIR = "scgrec_data/steam_data"
OUT_DIR = "scgrec_ready"
SCGREC_ID = "1F9kr_YWimBtexJEH-zkDzCOwl1q7GmFp"  # nota al pie 8 del paper SCGRec
os.makedirs(OUT_DIR, exist_ok=True)

if not os.path.isdir(DATA_DIR):
    print("No encuentro", DATA_DIR, "-> descargando de Drive...")
    import subprocess, sys, zipfile, tarfile
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "gdown"], check=False)
    import gdown
    raw = gdown.download(id=SCGREC_ID, output="scgrec_raw", quiet=False)
    if zipfile.is_zipfile(raw):
        zipfile.ZipFile(raw).extractall("scgrec_data")
    elif tarfile.is_tarfile(raw):
        tarfile.open(raw).extractall("scgrec_data")
    print("listo")
else:
    print("Usando datos ya extraídos en", DATA_DIR)

P = lambda *a: os.path.join(DATA_DIR, *a)

No encuentro scgrec_data/steam_data -> descargando de Drive...


Downloading...
From (original): https://drive.google.com/uc?id=1F9kr_YWimBtexJEH-zkDzCOwl1q7GmFp
From (redirected): https://drive.google.com/uc?id=1F9kr_YWimBtexJEH-zkDzCOwl1q7GmFp&confirm=t&uuid=19b84f41-3fbf-4264-a9d6-97967f7e23c6
To: /content/scgrec_raw
100%|██████████| 993M/993M [00:16<00:00, 58.5MB/s]
/tmp/ipykernel_2275/139386362.py:22: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tarfile.open(raw).extractall("scgrec_data")


listo


## INSPECCIÓN — formato crudo (separador por comas, user_id al inicio)

In [2]:
def head_raw(path, n=2, maxchars=300):
    print(f"\n### {path}")
    if not os.path.exists(path):
        print("  (no existe)"); return
    with open(path, encoding="utf-8", errors="replace") as f:
        for i, line in enumerate(f):
            if i >= n:
                break
            print(f"  L{i}: {line[:maxchars].rstrip()}")

for fn in ["App_ID_Info.txt", "Games_Genres.txt", "Games_Developers.txt",
           "Games_Publishers.txt", "train_game.txt", "train_time.txt",
           "test_data/test_game.txt", "test_data/test_time.txt"]:
    head_raw(P(*fn.split("/")))


### scgrec_data/steam_data/App_ID_Info.txt
  L0: 10,Counter-Strike,game,9.99,2000-11-01 00:00:00,-1,0,1
  L1: 20,Team Fortress Classic,game,4.99,1999-04-01 00:00:00,-1,0,1

### scgrec_data/steam_data/Games_Genres.txt
  L0: 10,Action
  L1: 20,Action

### scgrec_data/steam_data/Games_Developers.txt
  L0: 10,Valve
  L1: 20,Valve

### scgrec_data/steam_data/Games_Publishers.txt
  L0: 10,Valve
  L1: 20,Valve

### scgrec_data/steam_data/train_game.txt
  L0: 76561198059262283,50,380,80,300,320,30,100,620,70,10,130,280,500,730,420,20,550,360,40,400,340
  L1: 76561197962522126,10,60,130,50,40,70

### scgrec_data/steam_data/train_time.txt
  L0: 76561198059262283,58,0,0,0,0,569,0,0,807,0,0,0,0,29,0,0,0,0,0,0,0
  L1: 76561197962522126,81,\N,\N,\N,\N,\N

### scgrec_data/steam_data/test_data/test_game.txt
  L0: 76561197973222695,220,223530
  L1: 76561197962522126,30

### scgrec_data/steam_data/test_data/test_time.txt
  L0: 76561197973222695,0,0
  L1: 76561197962522126,\N


## CATEGORÍAS y METADATA (separados por COMAS, sin header)

In [3]:
def load_pairs(path, valcol):
    """app_id,valor por línea — partition() es robusto a comas dentro del valor."""
    a, v = [], []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")
            if not line:
                continue
            k, _, val = line.partition(",")
            a.append(int(k)); v.append(val)
    return pd.DataFrame({"app_id": a, valcol: v})


genres     = load_pairs(P("Games_Genres.txt"), "genre")
developers = load_pairs(P("Games_Developers.txt"), "developer")
publishers = load_pairs(P("Games_Publishers.txt"), "publisher")
app_info = pd.read_csv(P("App_ID_Info.txt"), sep=",", header=None,
                       names=["app_id", "name", "type", "price",
                              "release_date", "metascore", "col6", "col7"],
                       on_bad_lines="skip", engine="python")

print("géneros:", genres.shape, "| developers:", developers.shape,
      "| publishers:", publishers.shape, "| app_info:", app_info.shape)
print("# géneros únicos:", genres["genre"].nunique(),
      "| # developers:", developers["developer"].nunique(),
      "| # publishers:", publishers["publisher"].nunique())

game2genres = genres.groupby("app_id")["genre"].apply(list)
game2dev    = developers.groupby("app_id")["developer"].apply(list)
game2pub    = publishers.groupby("app_id")["publisher"].apply(list)
print("ejemplo géneros app 10:", game2genres.get(10))

géneros: (3842, 2) | developers: (2337, 2) | publishers: (2380, 2) | app_info: (2238, 8)
# géneros únicos: 22 | # developers: 1170 | # publishers: 689
ejemplo géneros app 10: ['Action']


## PARSER (streaming a parquet por lotes — no satura RAM)

In [4]:
SCHEMA = pa.schema([("user_id", pa.int64()), ("app_id", pa.int32()), ("playtime", pa.float32())])
BATCH = 4_000_000   # filas por lote antes de volcar a disco


def parse_split_to_parquet(game_path, time_path, out_path, max_lines=None):
    """Lee los splits de adyacencia (coma-sep, user_id al inicio, tiempos con \\N)
    y escribe parquet en lotes. Devuelve stats sin retener las interacciones."""
    writer = pq.ParquetWriter(out_path, SCHEMA)
    bu, bg, bt = array("q"), array("i"), array("f")
    n_rows = n_zero = n_users = 0
    games_seen = set()

    def flush():
        nonlocal bu, bg, bt
        if len(bu) == 0:
            return
        writer.write_table(pa.table({
            "user_id": pa.array(bu, pa.int64()),
            "app_id":  pa.array(bg, pa.int32()),
            "playtime": pa.array(bt, pa.float32()),
        }))
        bu, bg, bt = array("q"), array("i"), array("f")

    with open(game_path, encoding="utf-8") as fg, open(time_path, encoding="utf-8") as ft:
        for idx, (lg, lt) in enumerate(zip(fg, ft)):
            if max_lines and idx >= max_lines:
                break
            tg = lg.rstrip("\n").split(",")
            tt = lt.rstrip("\n").split(",")
            if not tg or tg[0] == "":
                continue
            uid = int(tg[0]); n_users += 1          # 1 línea = 1 usuario
            for g, t in zip_longest(tg[1:], tt[1:], fillvalue="\\N"):
                if g is None or g == "":
                    continue
                pt = 0.0 if t in (None, "\\N", "") else float(t)
                bu.append(uid); bg.append(int(g)); bt.append(pt)
                games_seen.add(int(g)); n_rows += 1
                if pt == 0.0:
                    n_zero += 1
            if len(bu) >= BATCH:
                flush()
    flush(); writer.close()
    return {"rows": n_rows, "users": n_users, "game_set": games_seen,
            "zero_pct": 100 * n_zero / max(1, n_rows)}


def print_stats(st, name):
    print(f"{name:6s}: {st['rows']:>12,} interac · {st['users']:>9,} usuarios · "
          f"{len(st['game_set']):>6,} juegos · playtime=0: {st['zero_pct']:.1f}%")


st_test  = parse_split_to_parquet(P("test_data", "test_game.txt"),
                                  P("test_data", "test_time.txt"),
                                  f"{OUT_DIR}/inter_test.parquet")
st_valid = parse_split_to_parquet(P("valid_data", "valid_game.txt"),
                                  P("valid_data", "valid_time.txt"),
                                  f"{OUT_DIR}/inter_valid.parquet")
print_stats(st_test, "test")
print_stats(st_valid, "valid")
print("ejemplo:", pd.read_parquet(f"{OUT_DIR}/inter_test.parquet").head(3).to_dict("records"))

test  :      116,314 interac ·    50,000 usuarios ·  2,115 juegos · playtime=0: 40.4%
valid :      116,314 interac ·    50,000 usuarios ·  2,110 juegos · playtime=0: 40.4%
ejemplo: [{'user_id': 76561197973222695, 'app_id': 220, 'playtime': 0.0}, {'user_id': 76561197973222695, 'app_id': 223530, 'playtime': 0.0}, {'user_id': 76561197962522126, 'app_id': 30, 'playtime': 0.0}]


## TRAIN (grande, streaming). MAX_TRAIN_LINES = nº usuarios; None = todos.

In [5]:
MAX_TRAIN_LINES = None   # pon p. ej. 200_000 para una pasada rápida

st_train = parse_split_to_parquet(P("train_game.txt"), P("train_time.txt"),
                                  f"{OUT_DIR}/inter_train.parquet",
                                  max_lines=MAX_TRAIN_LINES)
print_stats(st_train, "train")
total = st_train["rows"] + st_valid["rows"] + st_test["rows"]
print(f"\nTOTAL interacciones (train+valid+test): {total:,}  (paper SCGRec ≈ 95,4M)")

train :   95,208,806 interac · 3,908,744 usuarios ·  2,675 juegos · playtime=0: 40.2%

TOTAL interacciones (train+valid+test): 95,441,434  (paper SCGRec ≈ 95,4M)


## Tabla de categorías por juego + GUARDAR (usa solo el set de juegos)

In [6]:
all_games = pd.Index(sorted(st_train["game_set"] | st_valid["game_set"] | st_test["game_set"]))
cat = pd.DataFrame({"app_id": all_games})
cat["genres"]     = cat["app_id"].map(lambda g: game2genres.get(g, []))
cat["developers"] = cat["app_id"].map(lambda g: game2dev.get(g, []))
cat["publishers"] = cat["app_id"].map(lambda g: game2pub.get(g, []))
cat = cat.merge(app_info[["app_id", "name", "type", "price", "release_date", "metascore"]],
                on="app_id", how="left")
cat.to_parquet(f"{OUT_DIR}/game_categories.parquet", index=False)

print("catálogo:", cat.shape[0], "juegos")
print("cobertura géneros:", cat["genres"].map(len).gt(0).mean().round(3),
      "| dev:", cat["developers"].map(len).gt(0).mean().round(3),
      "| pub:", cat["publishers"].map(len).gt(0).mean().round(3))
print(f"\nGuardado en {OUT_DIR}/ -> inter_{{train,valid,test}}.parquet + game_categories.parquet")

catálogo: 2675 juegos
cobertura géneros: 0.751 | dev: 0.821 | pub: 0.837

Guardado en scgrec_ready/ -> inter_{train,valid,test}.parquet + game_categories.parquet


## Resumen del mapeo a CPGRec+

In [7]:
print("""
=== LISTO para reproducir CPGRec/CPGRec+ ===
  inter_train/valid/test.parquet : user_id · app_id · playtime  (colaborativo + dwelling time)
  game_categories.parquet        : app_id · genres · developers · publishers (grafos SGC/CNA) · name/price/release/metascore
  Señal social (friends/Groups)  : OMITIDA a propósito (diferenciación vs SCGRec)

Para entrenar: lee los parquet por chunks o con pyarrow dataset; NO hace falta
cargar los 95M en un DataFrame de una sola vez.
""")


=== LISTO para reproducir CPGRec/CPGRec+ ===
  inter_train/valid/test.parquet : user_id · app_id · playtime  (colaborativo + dwelling time)
  game_categories.parquet        : app_id · genres · developers · publishers (grafos SGC/CNA) · name/price/release/metascore
  Señal social (friends/Groups)  : OMITIDA a propósito (diferenciación vs SCGRec)

Para entrenar: lee los parquet por chunks o con pyarrow dataset; NO hace falta
cargar los 95M en un DataFrame de una sola vez.

